In [0]:
dbutils.widgets.removeAll()

In [0]:
# File Parameters
dbutils.widgets.text("ClientID","","") 
dbutils.widgets.text("FileID","","") 
dbutils.widgets.text("FileLayoutID","","") 
dbutils.widgets.text("FileLayoutDescription","","") 
dbutils.widgets.text("ColumnDelimiter","","") 
dbutils.widgets.text("HasHeader","","") 
dbutils.widgets.text("IgnoreHeader","","") 
dbutils.widgets.text("TextQualifier","","") 

# File to be Processed
dbutils.widgets.text("FullFileName","","")  

# Schema File
dbutils.widgets.text("SchemaFile","","")  

# Processed File 
dbutils.widgets.text("ProcessedPath","","") 

In [0]:
ClientId = dbutils.widgets.get("ClientID")
FileId = dbutils.widgets.get("FileID")
FileLayoutId = dbutils.widgets.get("FileLayoutID")
FileLayoutDescription = dbutils.widgets.get("FileLayoutDescription")
ColumnDelimiter = dbutils.widgets.get("ColumnDelimiter")
HasHeader = dbutils.widgets.get("HasHeader")
IgnoreHeader = dbutils.widgets.get("IgnoreHeader")
TextQualifier = dbutils.widgets.get("TextQualifier")
FullFileName = dbutils.widgets.get("FullFileName")
SchemaFile = dbutils.widgets.get("SchemaFile")
ProcessedPath = dbutils.widgets.get("ProcessedPath")


In [0]:
# Hardcoded debug variables matching the previous payload parameters
import os
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "../.."))

# Retrieve from widgets or fall back to defaults
ClientId = dbutils.widgets.get("ClientID") or "01"
FileId = dbutils.widgets.get("FileID") or "01"
FileLayoutId = dbutils.widgets.get("FileLayoutID") or "274"
FileLayoutDescription = dbutils.widgets.get("FileLayoutDescription") or ("ProviderHierarchy" if FileLayoutId == "274" else "Provider")
ColumnDelimiter = dbutils.widgets.get("ColumnDelimiter") or ","
HasHeader = dbutils.widgets.get("HasHeader") or "true"
IgnoreHeader = dbutils.widgets.get("IgnoreHeader") or "False"
TextQualifier = dbutils.widgets.get("TextQualifier") or "\""

if FileLayoutId == "274":
    FullFileName = f"{project_root}/temp/274/provider_hierarchy_nonsolo.csv"
    SchemaFile = f"{project_root}/DimProvider/Bronze/Schema/provider_hierarchy_7.12_schema.json"
    ProcessedPath = "/Volumes/provider_274/bronze/processed_parquet/provider_hierarchy"
else:
    FullFileName = f"{project_root}/temp/837/provider1.csv"
    SchemaFile = f"{project_root}/DimProvider/Bronze/Schema/provider_7.12_schema.json"
    ProcessedPath = "/Volumes/provider_274/bronze/processed_parquet/provider"

In [0]:
%run "../CommonMethods/Helpers/FileHandling"

In [0]:
%run "../CommonMethods/Helpers/SynJSONCreatorClass"

In [0]:
from pyspark.sql.types import StructType
from pyspark.sql.functions import lit, to_timestamp, current_timestamp

ErrorMessage = ""
doubleQuote = '"'

# get job id
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    currentJobId = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
except Exception:
    # serverless fallback - ctx.tags() is not whitelisted on serverless
    currentJobId = "Undefined"

In [0]:
def process_move_file(ClientId, FileId, FileLayoutId, FileLayoutDescription,
                      ColumnDelimiter, HasHeader, IgnoreHeader, textQualifier,
                      FullFileName, SchemaFile, ProcessedPath):
    rJSON = synJSONCreator()
    ErrorMessage = ""
    doubleQuote = '"'

    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        currentJobId = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
    except Exception:
        currentJobId = "Undefined"
    
    rJSON.addBraceStart()
    rJSON.addNewEntry("CurrentJobId", currentJobId)

    dfFile = spark.createDataFrame([], StructType([]))

    try:
        if IgnoreHeader == "False":
            dfFile = delimitedFile(FullFileName, SchemaFile, HasHeader,
                                   ColumnDelimiter, textQualifier)
        elif IgnoreHeader == "True" and HasHeader == "True":
            dfFile = isIgnoreHeader(FullFileName, SchemaFile, ColumnDelimiter,
                                    textQualifier)
        
        if len(dfFile.take(1)) > 0:
            filtered_cols = [col_name for col_name in dfFile.columns if not col_name.startswith("Filler_")]
            dfFile = dfFile.select(filtered_cols)

            dfFile = dfFile.withColumn("FILE_ID", lit(FileId)) \
                .withColumn("FILE_LAYOUT_ID", lit(FileLayoutId)) \
                .withColumn("FILE_LAYOUT_DESCRIPTION", lit(FileLayoutDescription)) \
                .withColumn("CLIENT_ID", lit(ClientId)) \
                .withColumn("LOAD_DATETIME", to_timestamp(current_timestamp(), "MM/dd/yyyy HH:mm:ss"))

            if ProcessedPath.count('.') == 2 and not ProcessedPath.startswith('/'):
                dfFile.write.format("delta").mode("append").saveAsTable(ProcessedPath)
            else:
                dfFile.write.format("parquet").mode("append").save(ProcessedPath)

            rJSON.addNewEntry("Status", "SUCCESS")
            rJSON.addNewEntry("ProcessedCount", str(dfFile.count()))
            rJSON.addNewEntry("ErrorMessage", "", newLine=False)

        else:
            rJSON.addNewEntry("Status", "SUCCESS")
            rJSON.addNewEntry("ProcessedCount", "0")
            rJSON.addNewEntry("ErrorMessage", "No records after filtering", newLine=False)

    except Exception as e:
        clean_err = str(e).strip().replace(doubleQuote, "").replace("\n", " ").replace("\r", " ").replace("\t", " ")
        rJSON.addNewEntry("Status", "FAILED")
        rJSON.addNewEntry("ProcessedCount", "0")
        rJSON.addNewEntry("ErrorMessage", clean_err, newLine=False) 

    rJSON.addBraceEnd()
    return rJSON.getJSON()


In [0]:
print(f"--- Starting Interactive Debug for File: {FileId} ---")

debug_result = process_move_file(
    ClientId=ClientId, 
    FileId=FileId, 
    FileLayoutId=FileLayoutId, 
    FileLayoutDescription=FileLayoutDescription,
    ColumnDelimiter=ColumnDelimiter, 
    HasHeader=HasHeader, 
    IgnoreHeader=IgnoreHeader, 
    textQualifier=TextQualifier,
    FullFileName=FullFileName, 
    SchemaFile=SchemaFile, 
    ProcessedPath=ProcessedPath
)

print("\n=== Processing Output Report ===")
print(debug_result)